# Clean the LILA Dataset

## Deal with Bounding Box

Here we deal with bounding boxes, ensuring that our dataset contains only images with bounding boxes. We also in the meantime produce the X, Y, W, H coordinates that will be required for our YOLO model.

In [85]:
import json

import polars
import polars as pl
from pathlib import Path
import numpy as np

from polars import json_normalize

annotations_path =  "community_fish_detection_dataset.json"

with open(annotations_path, "r") as f:
    annotations = json.load(f)

print(f"Image: {annotations["images"][0]}\n Annotations: {annotations['annotations'][0]}\n Categories: {annotations['categories'][0]}\n")
print(len(annotations["images"]), len(annotations["annotations"]), len(annotations["categories"]))

images=annotations["images"]
annotations=annotations["annotations"]
cleaned = {}
for col in annotations[0].keys():
    if col != "bbox":
        cleaned[col] = []

# Ensure that we have the necessary columns
cleaned["x"] = []
cleaned["y"] = []
cleaned["w"] = []
cleaned["h"] = []


bbox_missing = []

for ann in annotations:
    # if any(str(v).startswith("empty") for v in ann.values()):
    #     continue
    if "bbox" not in ann:
        bbox_missing.append(ann["image_id"])
        continue
    ann["x"], ann["y"], ann["w"], ann["h"] = ann["bbox"]
    del ann["bbox"]
    for col in ann:
        cleaned[col].append(ann[col])

print(len(cleaned), len(bbox_missing))

Image: {'id': 'torsi_20190716-021037.129.JPG', 'file_name': 'JPEGImages/torsi_20190716-021037.129.JPG', 'width': 3296, 'height': 2472, 'is_train': False, 'dataset': 'torsi', 'original_data_source': '20190716-021037.129.JPG'}
 Annotations: {'id': 0, 'image_id': 'torsi_20190716-021037.129.JPG', 'category_id': 1, 'bbox': [2590.0, 1773.0, 706.0, 574.0]}
 Categories: {'id': 1, 'name': 'fish'}

1903035 2167541 2
7 1232492


In [86]:
df = pl.from_dict(cleaned, strict=False)

id,image_id,category_id,x,y,w,h
str,str,i64,f64,f64,f64,f64
"""0""","""torsi_20190716-021037.129.JPG""",1,2590.0,1773.0,706.0,574.0
"""1""","""torsi_20190716-021038.179.JPG""",1,561.0,1922.0,467.0,550.0
"""2""","""torsi_20190716-021039.154.JPG""",1,731.0,537.0,459.0,936.0
"""3""","""torsi_20190716-021039.154.JPG""",1,474.0,105.0,1006.0,614.0
"""4""","""torsi_20190716-021040.129.JPG""",1,1222.0,341.0,537.0,915.0
…,…,…,…,…,…,…
"""valid/2019-10-22_17-57-46_righ…","""valid/2019-10-22_17-57-46_righ…",1,1283.43936,316.24992,636.56064,483.6996
"""valid/2019-10-22_17-57-46_righ…","""valid/2019-10-22_17-57-46_righ…",1,1321.40928,319.05954,598.59072,475.2702
"""valid/2019-10-22_17-57-46_righ…","""valid/2019-10-22_17-57-46_righ…",1,1359.3792,321.87942,560.6208,466.82028
